# Install required libraries

In [ ]:
!pip install klib lazypredict dask[dataframe] dill shap

# Save the state of the notebook

In [ ]:
# import dill
#
# dill.dump_session("data_analysis.pkl")

# Load the state of the notebook

In [ ]:
# import dill
#
# dill.load_session("data_analysis.pkl")

# read all the csv files in the data folder and concatenate them into one

In [ ]:
import os
import glob
import gc
import pandas as pd

path = "data/"
all_files = glob.glob(os.path.join(path, "*.csv"))

df_from_each_file = (pd.read_csv(f) for f in all_files)
df = pd.concat(df_from_each_file, ignore_index=True)
del df_from_each_file
gc.collect()

# Exploratory Data Analysis

## View basic information about the data

In [ ]:
from IPython.display import display
import numpy as np

display(df.info())
display(df.head())

## how many records does the data have?

In [ ]:
display("Number of records: ", df.shape[0])

## how many features does the data have?

In my project, I will consider all the features except the ' Label' as the
features.

In [ ]:
display("Number of features: ", df.shape[1] - 1)

## How many many different classes exist in the dataset?

Note that first we transform the 'Label' column from Object to String

In [ ]:
display("Number of classes: ", len(df[" Label"].unique()))
display("Number of examples per class:\n", df[" Label"].value_counts())

## View missing value information

In [ ]:
display("Total number of missing values", df.isnull().sum().sum())
display("Total number of missing values in each column", df.isnull().sum())
display(
    "percentage of missing values in each column", df.isnull().sum() / df.shape[0] * 100
)

# Which featiures are not numerical?

In [ ]:
display("Non numerical features:\n", df.select_dtypes(exclude=[np.number]).columns)

# Data Cleaning

## First we Clean the data by performing the following steps:

1. Drop empty or single valued columns

In [ ]:
df.dropna(axis=1, how="all", inplace=True)

df = df.loc[:, df.apply(pd.Series.nunique) != 1]

2. Drop empty rows

In [ ]:
df.dropna(axis=0, how="all", inplace=True)

3. Drop duplicate rows (wait)
3. (alternative) add a column to count the number of duplicates and then drop
   duplicates. This way we can keep track of the repeated rows.

In [ ]:
df[" frequency"] = df.groupby(df.columns.tolist(), sort=False).cumcount()
df[" frequency"] = df[" frequency"].replace(np.nan, 0)
tmp_df_len = df.shape[0]
df.drop_duplicates(inplace=True)
display("Number of removed duplicates: ", tmp_df_len - df.shape[0])

display(df.sort_values(" frequency", ascending=False))

4. Mark the values that are Infinity with -2 and NaN with -1

In [ ]:
df.replace([np.inf, -np.inf], -2, inplace=True)
df.replace(np.nan, -1, inplace=True)

5. Drop columns with more than 50% Zero (0) values

In [ ]:
# calculate the percentage of zeros in each column
zero_percentage = (df == 0).sum() / df.shape[0] * 100
display("Percentage of zeros in each column", zero_percentage)
zero_percentage = zero_percentage.drop(" frequency")  # we need the frequency column
columns_to_drop = zero_percentage[zero_percentage > 50].index.tolist()
display("Columns to drop", columns_to_drop)
df.drop(columns=columns_to_drop, inplace=True)

6. Drop columns with more than 30% (`-2` + `-1`) values

In [ ]:
# calculate the percentage of -2 and -1 in each column
missing_percentage = (df == -2).sum() / df.shape[0] * 100
missing_percentage += (df == -1).sum() / df.shape[0] * 100
display("Percentage of missing values in each column", missing_percentage)
df = df.loc[:, missing_percentage <= 30]

7. Drop rows with atleast one -1 or -2 values

In [ ]:
df = df[(df != -1).all(axis=1)]
df = df[(df != -2).all(axis=1)]

8. Transform the ' Label' column with LabelEncoder

In [ ]:
from sklearn.preprocessing import LabelEncoder

features = df.drop(columns=[" Label"])
label = df[" Label"]

le = LabelEncoder()
label = le.fit_transform(label)
display("Unique labels: ", le.classes_)
display("Mapped labels: ", le.transform(le.classes_))

9. Cast all the columns except the ' Label' column to float

In [ ]:
features = features.astype(float)

# Scaling

Data scaling allows the algorithm to converge faster and perform better.
We will use the StandardScaler to scale the data.

Note that we don't want to scale the target column ' Label', since
it is a categorical column.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
features = pd.DataFrame(scaler.fit_transform(features), columns=features.columns)

df = pd.concat([features, pd.DataFrame(label, columns=[" Label"])], axis=1)

# Visualization

## correlation matrix

In [ ]:
import klib

klib.corr_plot(df, annot=False)

## correlation to target

In [ ]:
klib.corr_plot(df, target=" Label")

## Data distribution

In [ ]:
klib.dist_plot(df[" Label"])

## High correlation filter

Here I can remove some of the data with high correlation, but I'm lazy and
I'm not sure if it's a good aproach for this dataset.

Here I only show high correlation features.

In [ ]:
df_tmp = df.copy()
label_encoder = LabelEncoder()
for column in df_tmp.select_dtypes(include=["object"]).columns:
    df_tmp[column] = label_encoder.fit_transform(df_tmp[column].astype(str))
df_tmp = (df_tmp - df_tmp.mean()) / df_tmp.std()
correlation = df_tmp.corr()

for column in correlation.columns:
    display(correlation[column].sort_values(ascending=False).head(2))

# Machine Learning Model

## Sample the data to reduce the training time for LazyPredict

I use 10% of the data to train the models and find the best model. The
stratify parameter is used to make sure that the classes are balanced.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(" Label", axis=1)
y = df[" Label"]

X_sample, _, y_sample, _ = train_test_split(
    X, y, train_size=0.1, random_state=42, stratify=y
)

## garbage collection

In [ ]:
del df_tmp
del zero_percentage
del missing_percentage
del le
del scaler
del correlation
del label_encoder
del tmp_df_len
del path
del all_files
del column
del label
del features
gc.collect()

## Use LazyPredict to find the best model

In [ ]:
from lazypredict.Supervised import LazyClassifier
import sklearn

X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=0.2)

ClassifierChain = [
  sklearn.ensemble._weight_boosting.AdaBoostClassifier,                         #AdaBoostClassifier
  sklearn.ensemble._bagging.BaggingClassifier,                                  #BaggingClassifier
  sklearn.naive_bayes.BernoulliNB,                                              #BernoulliNB
  # sklearn.calibration.CalibratedClassifierCV,                                 #CalibratedClassifierCV # Runs forever
  sklearn.naive_bayes.CategoricalNB,                                            #CategoricalNB
  sklearn.tree._classes.DecisionTreeClassifier,                                 #DecisionTreeClassifier
  sklearn.dummy.DummyClassifier,                                                #DummyClassifier
  sklearn.tree._classes.ExtraTreeClassifier,                                    #ExtraTreeClassifier
  sklearn.ensemble._forest.ExtraTreesClassifier,                                #ExtraTreesClassifier
  sklearn.model_selection._classification_threshold.FixedThresholdClassifier,   #FixedThresholdClassifier
  sklearn.naive_bayes.GaussianNB,                                               #GaussianNB
  sklearn.neighbors._classification.KNeighborsClassifier,                       #KNeighborsClassifier
  # sklearn.semi_supervised._label_propagation.LabelPropagation,                #LabelPropagation       # Fills  the ram
  # sklearn.semi_supervised._label_propagation.LabelSpreading,                  #LabelSpreading         # Fills  the ram
  sklearn.discriminant_analysis.LinearDiscriminantAnalysis,                     #LinearDiscriminantAnalysis
  sklearn.linear_model._logistic.LogisticRegression,                            #LogisticRegression
  sklearn.neighbors._nearest_centroid.NearestCentroid,                          #NearestCentroid
  sklearn.svm._classes.NuSVC,                                                   #NuSVC
  sklearn.linear_model._passive_aggressive.PassiveAggressiveClassifier,         #PassiveAggressiveClassifier
  sklearn.linear_model._perceptron.Perceptron,                                  #Perceptron
  sklearn.discriminant_analysis.QuadraticDiscriminantAnalysis,                  #QuadraticDiscriminantAnalysis
  sklearn.ensemble._forest.RandomForestClassifier,                              #RandomForestClassifier
  sklearn.linear_model._ridge.RidgeClassifier,                                  #RidgeClassifier
  sklearn.linear_model._ridge.RidgeClassifierCV,                                #RidgeClassifierCV
  sklearn.linear_model._stochastic_gradient.SGDClassifier,                      #SGDClassifier
  sklearn.semi_supervised._self_training.SelfTrainingClassifier,                #SelfTrainingClassifier
  sklearn.ensemble._stacking.StackingClassifier,                                #StackingClassifier
  sklearn.model_selection._classification_threshold.TunedThresholdClassifierCV, #TunedThresholdClassifierCV
  sklearn.svm._classes.LinearSVC,                                               #LinearSVC              #  Takes a long time to run
  # sklearn.svm._classes.SVC,                                                   #SVC                    # Runs forever
]

clf = LazyClassifier(verbose=9, classifiers=ClassifierChain)
models, predictions = clf.fit(X_train, X_test, y_train, y_test)

In [ ]:
display(models)

In [ ]:
display(predictions)

## Visualizing the model performance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(5, 10))
sns.set_theme(style="whitegrid")
ax = sns.barplot(y=models.index, x="Accuracy", data=models)
plt.xlim(0, 1)
plt.savefig("accuracy.png")

In [ ]:
plt.figure(figsize=(5, 10))
sns.set_theme(style="whitegrid")
ax = sns.barplot(y=models.index, x="Balanced Accuracy", data=models)
plt.xlim(0, 1)
plt.savefig("balanced_accuracy.png")

In [ ]:
plt.figure(figsize=(5, 10))
sns.set_theme(style="whitegrid")
ax = sns.barplot(y=models.index, x="F1 Score", data=models)
plt.xlim(0, 1)
plt.savefig("f1_score.png")

In [ ]:
plt.figure(figsize=(5, 10))
sns.set_theme(style="whitegrid")
ax = sns.barplot(y=models.index, x="Time Taken", data=models)
plt.xlim()
plt.savefig("time_taken.png")

## Split the data into training and testing with 80% training and 20% testing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## Train the 5 models

ExtraTreesClassifier ranked first
RandomForestClassifier ranked second
BaggingClassifier ranked fourth
GaussianNB ranked seventh
AdaBoostClassifier ranked seventeenth

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, BaggingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report
from yellowbrick.classifier import ClassificationReport, ConfusionMatrix
import numpy as np
import pandas as pd
import time
import dill


models = {
    "ExtraTreesClassifier": ExtraTreesClassifier(),
    "RandomForestClassifier": RandomForestClassifier(),
    "BaggingClassifier": BaggingClassifier(),
    "GaussianNB": GaussianNB(),
    "AdaBoostClassifier": AdaBoostClassifier(),
}

models_info_dict = {}

# Train models and collect performance metrics
for model_name, model in models.items():
    start_time = time.time()
    model.fit(X_train, y_train)
    end_time = time.time()
    train_time = end_time - start_time

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)

    models_info_dict[model_name] = {
        "accuracy": accuracy,
        "train_time": train_time,
        "classification_report": report,
    }
    # save the model
    with open(f"{model_name}.pkl", "wb") as f:
        dill.dump(model, f)

### plotting

Extract data from models_info_dict

In [ ]:
model_names = list(models_info_dict.keys())
accuracies = [models_info_dict[model]['accuracy'] for model in model_names]
train_times = [models_info_dict[model]['train_time'] for model in model_names]

### 1. Accuracy Comparison Bar Plot

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=accuracies, y=model_names, palette='viridis')
plt.title('Model Accuracy Comparison')
plt.xlabel('Accuracy')
plt.ylabel('Models')
plt.show()
plt.savefig("accuracy_comparison.png")

#### 2. Training Time Comparison Bar Plot

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=train_times, y=model_names, palette='magma')
plt.title('Model Training Time Comparison')
plt.xlabel('Training Time (seconds)')
plt.ylabel('Models')
plt.show()
plt.savefig("training_time_comparison.png")

#### 3. Precision, Recall, F1-Score Heatmaps

In [ ]:
def plot_classification_report(report, title):
    df_report = pd.DataFrame(report).transpose().drop(['support'], axis=1)
    plt.figure(figsize=(8, 6))
    sns.heatmap(df_report.iloc[:-1, :], annot=True, cmap='coolwarm', fmt='.2f')
    plt.title(title)
    plt.show()
    plt.savefig(f"{title}.png")


for model_name, model_info in models_info_dict.items():
    plot_classification_report(model_info['classification_report'], f'{model_name} Classification Report')

#### 4. Radar Chart

In [ ]:
from math import pi


def plot_radar_chart(models_info_dict):
    categories = ['accuracy', 'precision', 'recall', 'f1-score']
    num_vars = len(categories)

    for model_name, model_info in models_info_dict.items():
        report = model_info['classification_report']['weighted avg']
        values = [model_info['accuracy'], report['precision'], report['recall'], report['f1-score']]
        values += values[:1]  # Repeat the first value to close the circle

        angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
        angles += angles[:1]

        plt.figure(figsize=(8, 8))
        ax = plt.subplot(111, polar=True)
        plt.xticks(angles[:-1], categories, color='grey', size=12)
        ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_name)
        ax.fill(angles, values, alpha=0.4)
        plt.title(f'{model_name} Performance Radar Chart', size=16)
        plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
        plt.show()
        plt.savefig(f"{model_name}_radar_chart.png")

#### 5. Confusion Matrix Visualization

In [ ]:
from sklearn.metrics import confusion_matrix
import itertools


def plot_confusion_matrix(y_true, y_pred, classes, title='Confusion matrix', cmap=plt.cm.Blues):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt), horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.savefig(f"{title}.png")


# Plot confusion matrices for each model
for model_name, model in models.items():
    y_pred = model.predict(X_test)
    plot_confusion_matrix(y_test, y_pred, classes=np.unique(y_test), title=f'{model_name} Confusion Matrix')

# Plot radar charts for each model
plot_radar_chart(models_info_dict)

## 5-fold cross validation

NOTE: This step is commented out because it takes a long time to run.

In [ ]:
# from sklearn.model_selection import cross_val_score
#
#
# for model_name, model in models.items():
#     scores = cross_val_score(model, X, y, cv=5)
#     print(f"{model_name}: {scores.mean()}")

# 4 best features from dataset overall

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

# Scale the data again using the min-max scaler since chi2 requires 
# non-negative values
scaler = MinMaxScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

bestfeatures = SelectKBest(score_func=chi2, k=10)
fit = bestfeatures.fit(X, y)
dfscores = pd.DataFrame(fit.scores_)
dfcolumns = pd.DataFrame(X.columns)

featureScores = pd.concat([dfcolumns, dfscores], axis=1)
featureScores.columns = ["Specs", "Score"]
display(featureScores.nlargest(4, "Score"))

## 4 best features from dataset based on each class

NOTE: This step is commented out because it takes a long time to run.

In [ ]:
# import shap
# from sklearn.ensemble import RandomForestClassifier
#
#
# model = RandomForestClassifier(random_state=42)
# model.fit(X, y)
# explainer = shap.TreeExplainer(model)
# shap_values = explainer.shap_values(X)
#
# for i, class_name in enumerate(model.classes_):
#     print(f"\nTop 4 features for {class_name}:")
#     shap.summary_plot(shap_values[i], X, plot_type="bar", max_display=4)